In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [2]:
train_dir = r'D:\new_prostate_vis\train'
test_dir =  r'D:\new_prostate_vis\test'
valid_dir = r'D:\new_prostate_vis\Valid'

In [3]:
import tensorflow.keras.backend as K 

from tensorflow.keras.models import Model

from tensorflow.keras.layers import *
import collections
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import regularizers
from  tensorflow.keras.initializers import *

In [4]:
batch_size = 4
img_height = 256
img_width = 256
no_of_classes = 4
classes_name = [0,1,2,3]
input_shape = (img_height , img_width , 3)
datagen = ImageDataGenerator(
 rescale=1. / 255,
 featurewise_center=True,
 horizontal_flip = False,
 vertical_flip = False,
 #validation_split = 0.1,
 featurewise_std_normalization=True)
train_generator = datagen.flow_from_directory(
 train_dir,
 target_size=(img_height, img_width),
 batch_size=batch_size,
 shuffle = True,
 class_mode='categorical')
validation_generator = datagen.flow_from_directory(
 valid_dir,
 target_size=(img_height, img_width),
 batch_size=batch_size,
 shuffle = True,
 class_mode='categorical')
# print(train_generator[0])
print("Trainging classes")
print(train_generator.class_indices)
print("Trainging Labels")
print(train_generator.labels)
print("Validation classes")
print(validation_generator.class_indices)
print("Validation Labels")
print(validation_generator.labels)

Found 16000 images belonging to 4 classes.
Found 4000 images belonging to 4 classes.
Trainging classes
{'0': 0, '1': 1, '2': 2, '3': 3}
Trainging Labels
[0 0 0 ... 3 3 3]
Validation classes
{'0': 0, '1': 1, '2': 2, '3': 3}
Validation Labels
[0 0 0 ... 3 3 3]


In [5]:
def conv_block(filter,x):
  b0=Conv2D(filter,(3,3),padding='same')(x)
  b1=BatchNormalization()(b0)
  b2=Activation('relu')(b1)
  return b2

def conv_block1(filter,x):
  b0=SeparableConv2D(filter,(3,3),padding='same')(x)
  b1=BatchNormalization()(b0)
  b2=Activation('relu')(b1)
  return b2

def residual_block(filter,x):
  w,h,c = x.shape[1],x.shape[2],x.shape[3]
  r0=AveragePooling2D(pool_size=(w,h))(x)
  r0=Conv2D(filter,(1,1),activation='sigmoid',padding='same')(r0)
  r1=Multiply()([x,r0])
  return r1

def scr_block(x,y,filters,gr):
  _,width,height,channel=list(x.shape)
  get_channel=channel//gr
  x=Reshape([width,height,get_channel,gr])(x)
  x=Permute([1,2,4,3])(x)
   
  _,width,height,channel=list(y.shape)
  get_channel=channel//gr
  y=Reshape([width,height,get_channel,gr])(y)
  y=Permute([1,2,4,3])(y)
  
  for i in range(gr):
        if i != 0 and i!=(gr-1):
            concat1 = y[:,:,:,0]
            for j in range(1,i):
                concat1 = Concatenate()([concat1,y[:,:,:,j]])
            concat1 = Concatenate()([concat1,x[:,:,:,i]])
            for j in range(i+1,gr):
                concat1 = Concatenate()([concat1,x[:,:,:,j]])
        elif i == (gr-1):
            concat1 = y[:,:,:,0]
            for j in range(1,i):
                concat1 = Concatenate()([concat1,y[:,:,:,j]])
            concat1 = Concatenate()([concat1,x[:,:,:,i]])
        else:
            concat1 = x[:,:,:,0]
            for j in range(1,gr):
                concat1 = Concatenate()([concat1,y[:,:,:,j]])  
        # conv3 = Conv2D(channel,1,padding='same',activation='relu')(concat1)

        # conv3 = conv_block(channel,concat1)
        res_x = residual_block(channel,concat1)
        if i==0:
            x_out = res_x
        else:
            x_out = Concatenate()([x_out,res_x])
  x_out = Conv2D(channel,1,padding='same')(x_out)
  return x_out

def resnet_block(block_input,filters,gr):
  conv1=conv_block(filters,block_input)
  conv2=scr_block(block_input,conv1,filters,gr)
  sum=Add()([conv2,block_input])
  res_out=Activation('relu')(sum)
  return res_out

def rccgnet():
  input = Input(shape=(256,256,3))
  conv1 = conv_block(16,input)
  scr1 = resnet_block(conv1,16,2)
  scr2 = resnet_block(conv1,16,4)
  scr3 = resnet_block(conv1,16,8)
  Add1 = Add()([scr1,scr2,scr3])
  pool1 = AveragePooling2D((2,2),(2,2))(Add1)
  conv2 = conv_block(32,pool1)
  scr4 = resnet_block(conv2,32,2)
  scr5 = resnet_block(conv2,32,4)
  scr6 = resnet_block(conv2,32,8)
  Add2 = Add()([scr4,scr5,scr6])
  pool2 = MaxPooling2D((2,2),(2,2))(Add2)
  conv3 = conv_block(64,pool2)
  scr7 = resnet_block(conv3,64,2)
  scr8 = resnet_block(conv3,64,4)
  scr9 = resnet_block(conv3,64,8)
  Add3 = Add()([scr7,scr8,scr9])
  pool3 = GlobalAveragePooling2D()(Add3)
  output = Dense(4, activation='softmax')(pool3)
  model = Model(inputs = input, outputs = output)
  return model

In [6]:
model = rccgnet()
model.compile(optimizer = 'Adam' , loss = 'categorical_crossentropy' , metrics = ["acc"])
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 256, 256, 3) 0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 256, 256, 16) 448         input_1[0][0]                    
__________________________________________________________________________________________________
batch_normalization (BatchNorma (None, 256, 256, 16) 64          conv2d[0][0]                     
__________________________________________________________________________________________________
activation (Activation)         (None, 256, 256, 16) 0           batch_normalization[0][0]        
______________________________________________________________________________________________

In [7]:
import time

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor = 'val_acc' , mode='max' ,
                                                  factor = 0.5 , patience = 10 , verbose=1 , cooldown = 1,
                                                 min_delta = 0.0001)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_acc', min_delta=0.0001, patience=30, verbose=1,
                                              mode = 'max', restore_best_weights = True)
check_path = 'F:/vishnu/weights/new10.h5'
checkpoint = tf.keras.callbacks.ModelCheckpoint(check_path, monitor = 'val_acc', verbose=1, save_best_only=True, save_weights_only=True, mode='max')

t10 = time.time()
history_1 = model.fit(x=train_generator , validation_data = validation_generator ,
                                  steps_per_epoch= len(train_generator) ,
                                  validation_batch_size = len(validation_generator)
                                  ,epochs = 100,callbacks = [reduce_lr, early_stop, checkpoint] )
t11 = time.time()
time_forloop = t11 - t10
print(time_forloop)

C:\Users\NITK\anaconda3\envs\Shyam_Lal\lib\site-packages\keras_preprocessing\image\image_data_generator.py:720: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn('This ImageDataGenerator specifies '
C:\Users\NITK\anaconda3\envs\Shyam_Lal\lib\site-packages\keras_preprocessing\image\image_data_generator.py:728: UserWarning: This ImageDataGenerator specifies `featurewise_std_normalization`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn('This ImageDataGenerator specifies '


Epoch 1/100
4000/4000 [==============================] - 769s 187ms/step - loss: 0.6872 - acc: 0.6983 - val_loss: 0.8004 - val_acc: 0.6438

Epoch 00001: val_acc improved from -inf to 0.64375, saving model to F:/vishnu/weights\new10.h5
Epoch 2/100
4000/4000 [==============================] - 734s 183ms/step - loss: 0.4756 - acc: 0.7878 - val_loss: 0.5033 - val_acc: 0.7692

Epoch 00002: val_acc improved from 0.64375 to 0.76925, saving model to F:/vishnu/weights\new10.h5
Epoch 3/100
4000/4000 [==============================] - 730s 182ms/step - loss: 0.4076 - acc: 0.8142 - val_loss: 0.4796 - val_acc: 0.7850

Epoch 00003: val_acc improved from 0.76925 to 0.78500, saving model to F:/vishnu/weights\new10.h5
Epoch 4/100
4000/4000 [==============================] - 730s 183ms/step - loss: 0.3687 - acc: 0.8341 - val_loss: 1.9416 - val_acc: 0.4930

Epoch 00004: val_acc did not improve from 0.78500
Epoch 5/100
4000/4000 [==============================] - 731s 183ms/step - loss: 0.3413 - acc: 0.84

4000/4000 [==============================] - 740s 185ms/step - loss: 0.0535 - acc: 0.9808 - val_loss: 0.1155 - val_acc: 0.9682

Epoch 00041: val_acc improved from 0.96525 to 0.96825, saving model to F:/vishnu/weights\new10.h5
Epoch 42/100
4000/4000 [==============================] - 742s 186ms/step - loss: 0.0464 - acc: 0.9827 - val_loss: 0.1311 - val_acc: 0.9535

Epoch 00042: val_acc did not improve from 0.96825
Epoch 43/100
4000/4000 [==============================] - 748s 187ms/step - loss: 0.0466 - acc: 0.9836 - val_loss: 0.2299 - val_acc: 0.9355

Epoch 00043: val_acc did not improve from 0.96825
Epoch 44/100
4000/4000 [==============================] - 746s 187ms/step - loss: 0.0444 - acc: 0.9857 - val_loss: 0.7547 - val_acc: 0.8325

Epoch 00044: val_acc did not improve from 0.96825
Epoch 45/100
4000/4000 [==============================] - 747s 187ms/step - loss: 0.0470 - acc: 0.9832 - val_loss: 0.1192 - val_acc: 0.9700

Epoch 00045: val_acc improved from 0.96825 to 0.97000, savin

In [7]:
model.load_weights('F:/vishnu/weights/new10.h5')

In [8]:
test_d = ImageDataGenerator(rescale=1. / 255)
test = test_d.flow_from_directory(
    test_dir,
    target_size=(256,256),
    batch_size=1,
    shuffle = False,
    class_mode='categorical')

Found 5000 images belonging to 4 classes.


In [ ]:
import numpy as np
test_step = test.n//test.batch_size
test.reset()
pred = model.predict_generator(test , steps = test_step , verbose = 1)
pred_class_indices = np.argmax(pred,axis=1)

## printing predicted labels
print(pred_class_indices)

C:\Users\NITK\anaconda3\envs\Shyam_Lal\lib\site-packages\tensorflow\python\keras\engine\training.py:2001: UserWarning: `Model.predict_generator` is deprecated and will be removed in a future version. Please use `Model.predict`, which supports generators.
  warnings.warn('`Model.predict_generator` is deprecated and '


3677/5000 [=====================>........] - ETA: 51s

In [ ]:
from sklearn.metrics import *
classes = [0,1,2,3]


for cl in classes:

    print("class: ",cl)

    a1 = np.uint8(test.labels == cl)
    a2 = np.uint8(pred_class_indices == cl)

    print('Accuracy {}'.format(accuracy_score(y_true=a1, y_pred=a2)))
    print('F1 {}'.format(f1_score(y_true=a1, y_pred=a2)))
    print('precision {}'.format(precision_score(y_true=a1, y_pred=a2)))
    print('recall {}'.format(recall_score(y_true=a1, y_pred=a2)))

    print('jaccard {}'.format(jaccard_score(y_true=a1, y_pred=a2)))
    print("_______________________________")

In [ ]:
print('Accuracy {}'.format(accuracy_score(y_true=test.labels, y_pred=pred_class_indices)))
print('F1 {}'.format(f1_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('precision {}'.format(precision_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('recall {}'.format(recall_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))

print('jaccard {}'.format(jaccard_score(y_true=test.labels, y_pred=pred_class_indices,average = "macro")))
print('confusion_matrix\n {}'.format(confusion_matrix(y_true=test.labels, y_pred=pred_class_indices)))
print('classification_report\n {}'.format(classification_report(y_true=test.labels, y_pred=pred_class_indices)))
print('\n\n')